# 06. Test Evaluation Across FreeSurfer Versions

This notebook evaluates FS-version-specific checkpoints (v5/v6/v7/v8) on their matching test sets,
using the cached-module evaluation utilities and shared FS split logic.


In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from segmentation.config import DEVICE, PATCH_SIZE, SEED, build_runtime_cfgs
from segmentation.api import prepare_data_pipeline
from segmentation.model import TransUNet3D
from segmentation.metrics import build_eval_plot_bundle, collect_test_metrics_fast_cached
from segmentation.plot import plot_region_dice, plot_tissue_dice
from segmentation.fs_experiments import (
    build_v8_split_from_root,
    build_versioned_splits_from_tensors,
    load_fs_version_map,
)

print(f"device={DEVICE} | seed={SEED}")


device=cuda | seed=1337


In [2]:
FS_VERSION_JSON = Path("fs_version_by_dataset_updated.json")
TENSORS_ROOT = Path("tensors")
GCLOUD_TENSORS_ROOT = Path("tensors_gcloud")
SPLIT_RATIOS = (0.8, 0.1, 0.1)

CKPT_ROOTS = {
    5: Path("checkpoints_fsv5"),
    6: Path("checkpoints_fsv6"),
    7: Path("checkpoints_fsv7"),
    8: Path("checkpoints_fsv8"),
}

if not FS_VERSION_JSON.exists():
    raise FileNotFoundError(f"Missing version map JSON: {FS_VERSION_JSON}")


FileNotFoundError: Missing version map JSON: fs_version_by_dataset_updated.json

In [ ]:
fs_major_by_dataset = load_fs_version_map(FS_VERSION_JSON)

splits_567 = build_versioned_splits_from_tensors(
    tensors_root=TENSORS_ROOT,
    fs_major_by_dataset=fs_major_by_dataset,
    seed=int(SEED),
    ratios=SPLIT_RATIOS,
    target_majors=(5, 6, 7),
)

split_v8 = build_v8_split_from_root(
    tensors_root=GCLOUD_TENSORS_ROOT,
    seed=int(SEED),
    ratios=SPLIT_RATIOS,
)

split_by_major = {
    5: splits_567[5],
    6: splits_567[6],
    7: splits_567[7],
    8: split_v8,
}

rows=[]
for major in (5,6,7,8):
    s=split_by_major[major]
    rows.append({
        "fs_major": major,
        "n_train": len(s["x_train"]),
        "n_val": len(s["x_val"]),
        "n_test": len(s["x_test"]),
    })

display(pd.DataFrame(rows).sort_values("fs_major").reset_index(drop=True))


In [ ]:
def find_latest_checkpoint(root: Path) -> Path | None:
    if not root.exists():
        return None
    cands = [p for p in root.rglob("transunet3d_best.pt") if p.is_file()]
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: p.stat().st_mtime)
    return cands[-1]


def evaluate_checkpoint_on_split(fs_major: int, ckpt_path: Path, split: dict):
    data_overrides = {
        "x_train_files": list(split["x_train"]),
        "y_train_files": list(split["y_train"]),
        "x_val_files": list(split["x_val"]),
        "y_val_files": list(split["y_val"]),
        "x_test_files": list(split["x_test"]),
        "y_test_files": list(split["y_test"]),
        "group_split_enabled": True,
        "cache_dir": str(ckpt_path.parent / "eval_cache_preproc"),
        "cache_rebuild": False,
    }
    train_overrides = {
        "effective_batch_size": 1,
        "initial_micro_batch_size": 1,
        "num_workers": 4,
        "prefetch_factor": 2,
        "spatial_window": None,
        "spatial_stride": None,
        "enforce_global_attention": True,
    }

    # Use model/training metadata from the checkpoint when available.
    ckpt = torch.load(ckpt_path, map_location="cpu")
    ckpt_model_cfg = dict(ckpt.get("model_cfg", {}))

    data_cfg, model_cfg, train_cfg, _ = build_runtime_cfgs(
        data_overrides=data_overrides,
        model_overrides=(ckpt_model_cfg if ckpt_model_cfg else None),
        train_overrides=train_overrides,
    )

    pipeline = prepare_data_pipeline(
        data_cfg=data_cfg,
        train_cfg=train_cfg,
        seed=int(SEED),
        device=DEVICE,
        patch_size=tuple(int(v) for v in PATCH_SIZE),
    )

    model = TransUNet3D(
        n_classes=pipeline.n_classes,
        in_channels=int(model_cfg["in_channels"]),
        base_shape=tuple(int(v) for v in pipeline.base_volume_shape),
        patch_size=tuple(int(v) for v in PATCH_SIZE),
        channels=tuple(int(v) for v in model_cfg["channels"]),
        transformer_depth=int(model_cfg["transformer_depth"]),
        n_heads=int(model_cfg["n_heads"]),
        dropout=float(model_cfg["dropout"]),
        positional_encoding=str(model_cfg.get("positional_encoding", "learned")),
    ).to(DEVICE)

    model_state = ckpt.get("model_state", ckpt)
    model.load_state_dict(model_state, strict=False)
    model.eval()

    sample_metrics_df, region_metrics_df, timing_df = collect_test_metrics_fast_cached(
        model=model,
        pipeline=pipeline,
        class_values=pipeline.class_values,
        patch_chunk_size=int(ckpt.get("runtime_patch_chunk", 192)),
        compute_boundary=False,
        region_df=None,
        device=DEVICE,
        results_cache_dir=(ckpt_path.parent / "eval_cache"),
        reuse_results_cache=True,
    )

    bundle = build_eval_plot_bundle(
        sample_metrics_df=sample_metrics_df,
        region_metrics_df=region_metrics_df,
        method="model",
        tissue_as_percent=True,
    )

    return {
        "fs_major": fs_major,
        "ckpt_path": ckpt_path,
        "pipeline": pipeline,
        "sample_metrics_df": sample_metrics_df,
        "region_metrics_df": region_metrics_df,
        "timing_df": timing_df,
        "bundle": bundle,
    }


In [ ]:
eval_results = {}
summary_rows = []

for major in (5, 6, 7, 8):
    split = split_by_major[major]
    ckpt = find_latest_checkpoint(CKPT_ROOTS[major])

    if ckpt is None:
        print(f"[skip fsv{major}] no checkpoint found under {CKPT_ROOTS[major]}")
        continue
    if len(split["x_test"]) == 0:
        print(f"[skip fsv{major}] empty test split")
        continue

    out = evaluate_checkpoint_on_split(major, ckpt, split)
    eval_results[major] = out

    sample_df = out["sample_metrics_df"]
    model_rows = sample_df[sample_df["method"] == "model"]
    summary_rows.append({
        "fs_major": major,
        "checkpoint": str(ckpt),
        "n_test": int(len(split["x_test"])),
        "mean_dice_fg": float(model_rows["mean_dice_fg"].mean()) if len(model_rows) else float("nan"),
        "dice_csf": float(model_rows["dice_csf"].mean()) if len(model_rows) else float("nan"),
        "dice_gm": float(model_rows["dice_gm"].mean()) if len(model_rows) else float("nan"),
        "dice_wm": float(model_rows["dice_wm"].mean()) if len(model_rows) else float("nan"),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("fs_major").reset_index(drop=True)
display(summary_df)


In [ ]:
if not eval_results:
    raise RuntimeError("No FS-version evaluation results available.")

majors = sorted(eval_results.keys())
fig, axes = plt.subplots(
    nrows=2,
    ncols=len(majors),
    figsize=(7 * len(majors), 14),
    constrained_layout=True,
)
if len(majors) == 1:
    axes = axes.reshape(2, 1)

for col, major in enumerate(majors):
    bundle = eval_results[major]["bundle"]
    ax_region = axes[0, col]
    plot_region_dice(bundle, ax=ax_region)
    ax_region.set_title(f"FS v{major}: Region Dice")

    ax_tissue = axes[1, col]
    plot_tissue_dice(bundle, ax=ax_tissue)
    ax_tissue.set_title(f"FS v{major}: Tissue Dice")

out_dir = Path("comparison_plots") / time.strftime("%Y%m%d_%H%M%S")
out_dir.mkdir(parents=True, exist_ok=False)
fig_path = out_dir / "test_eval_fs_versions.png"
fig.savefig(fig_path, dpi=200)
summary_df.to_csv(out_dir / "test_eval_fs_versions_summary.csv", index=False)

print(f"saved figure -> {fig_path}")
print(f"saved summary -> {out_dir / 'test_eval_fs_versions_summary.csv'}")
